# Salting — How to Explain It in an Interview (3 Years Experience)

## 1. One-liner (say this first, ~15 seconds)

> **"Salting is a technique to fix data skew by adding a random prefix ('salt') to skewed keys before a join or groupBy, spreading hot keys across many tasks, then removing the salt afterwards."**

---

## 2. The Problem — show you've hit this in production, not just read about it

> "I hit this in a project where we joined a large transactions table with a customer dimension. Most Spark stages finished in minutes, but the job always **hung at 99%** — the Spark UI showed one shuffle task taking 40+ minutes while its siblings finished in seconds. The classic **data-skew signature**. In our case, a huge number of rows had `customer_id = 'NULL'` (guest accounts), and since Spark partitions by the hash of the join key, ALL those rows landed in one task."

---

## 3. How You Diagnosed It — this is what separates 3-YOE from a fresher

> "I confirmed it in the **Spark UI** by looking at the **shuffle read sizes and task durations per executor** — one task was reading GBs while the rest read MBs. If the skew is in an aggregation, the same thing shows up in one task doing a huge share of the shuffle write."

---

## 4. The Fix — whiteboard it if allowed

> "Salting: pick **N** (like 8 or 10), add a random salt to the hot key on the big side — the key becomes `(key, salt)`. That splits the one huge key into N buckets so rows spread across N partitions and parallelize. But for a join, the other side must be **replicated N times** (each row exploded with salt `0..N-1`) so every salted row on the big side still finds its match. **Join on the salted key, then group by the original key to undo it.**"

---

## 5. Trade-offs and Alternatives — senior-sounding close

> "Before salting manually, I'd always check the simpler options first:
>
> 1. **Broadcast join** if the other table is small — often the skew disappears entirely because there's no shuffle at all.
> 2. **Spark 3.x AQE skew-join handling** — with `spark.sql.adaptive` enabled (default on Databricks), it splits skewed partitions automatically.
> 3. **Salting only the KNOWN hot keys** (e.g., NULLs, one mega-city), so you don't pay the N-times replication cost on the whole table.
>
> Salting trades extra shuffle volume (small table duplicated N times) for even task parallelism — it's the most manual option, so I reach for it when broadcast and AQE can't solve it."

---

## Demo Code (in case they ask you to write it)

```python
from pyspark.sql import functions as F

N = 8  # number of salt buckets

big_df = spark.range(0, 100_000) \
    .withColumn("key", F.when(F.col("id") % 10 == 0, "HOT_KEY")
                         .otherwise(F.concat("key_", (F.col("id") % 100).cast("string"))))

small_df = spark.createDataFrame([("HOT_KEY", "val"), ("key_1", "val1"), ("key_2", "val2")],
                                  ["key", "value"])

# 1) Salt the skewed side: hot key gets spread into N buckets
big_salted = big_df.withColumn(
    "salt",
    F.when(F.col("key") == "HOT_KEY", (F.rand() * N).cast("int"))  # salt only the hot key
     .otherwise(F.lit(-1))
).withColumn("key_salted", F.concat_ws("_", "key", "salt"))

# 2) Replicate the small table N times so every salted bucket finds its match
salt_range = spark.range(0, N).withColumnRenamed("id", "salt")
small_salted = small_df.join(salt_range, "salt") \
    .withColumn("key_salted", F.concat_ws("_", "key", "salt"))

# 3) Join on the salted key, then aggregate by the ORIGINAL key
result = big_salted.join(small_salted, "key_salted") \
    .groupBy("key") \
    .count()

display(result.orderBy(F.desc("count")))
```

---

## Key Sound Bite to Close the Interview Answer

> **"Salting trades a bit of shuffle volume for even task parallelism — but I'd first check if a broadcast join or AQE already solves it, because salting manually is the most invasive option."**

In [0]:
#Salt the large/skewed dataset:
from pyspark.sql.functions import floor, rand

df1_salted = df1.withColumn("salt", floor(rand() * 5))  # 5 buckets


In [0]:
#Replicate the small dataset:
from pyspark.sql.functions import explode, array, lit

salt_values = array([lit(i) for i in range(5)])
df2_expanded = df2.withColumn("salt", explode(salt_values))


In [0]:
# Join on (key, salt):
joined = df1_salted.join(df2_expanded, ["customer_id", "salt"])


In [0]:
# Drop Salt
result = joined.drop("salt")
